# Figure 1b: HAPS Free-Space Path Loss Comparison

This notebook generates the comparison of free-space path loss for the HAPS/UAV links (matching `generate_pathloss_table.py`'s convention):

1. **Service link**: HAPS (20 km) -> UE (1.5 m), 2 GHz, swept 0-100 km horizontal
2. **Relay link, 1st hop**: HAPS (20 km) -> UAV (500 m AGL), 38 GHz -- a fixed, near-vertical hop (~19.5 km slant range); the UAV sits essentially below the HAPS, so this leg does not sweep with ground distance
3. **Relay link, 2nd hop**: UAV (500 m AGL) -> UE (1.5 m), 2 GHz, swept 0-100 km horizontal as the UAV relays outward toward the UE
4. **Combined relay path**: HAPS -> UAV -> UE, the dB sum of hops 1 and 2

All distances use the true 3D slant distance (horizontal distance plus the
altitude difference between the two endpoints), per Equation (1):

$$L_{FSPL} = 32.44 + 20\log_{10}(f_{MHz}) + 20\log_{10}(d_{3D,km}) \[dB\]$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Equation (1): Free-Space Path Loss

The function accepts frequency in MHz and the 3D slant distance in km
(horizontal distance and the tx/rx altitude difference combined), matching
`haps_a2g_pathloss_db()` in `channel_model.py`.

In [ ]:
def fspl_db(f_mhz, d_km):
    """Calculate free-space path loss in dB given a 3D distance."""
    return 32.44 + 20 * np.log10(f_mhz) + 20 * np.log10(d_km)


def slant_dist_km(r_horiz_km, h_tx_m, h_rx_m):
    """3D slant distance [km] between a horizontal ground offset and two endpoint altitudes [m]."""
    dh_km = (h_tx_m - h_rx_m) / 1000.0
    return np.sqrt(r_horiz_km ** 2 + dh_km ** 2)

## Parameters and calculations

In [ ]:
r_horiz_km = np.linspace(0, 100, 500)
f_2ghz = 2000
f_38ghz = 38000

HAPS_ALT_M = 20_000.0
UAV_ALT_M = 500.0
UE_ALT_M = 1.5

d_service_km = slant_dist_km(r_horiz_km, HAPS_ALT_M, UE_ALT_M)   # HAPS -> UE

# HAPS -> UAV is a fixed, near-vertical hop: the UAV sits essentially below
# the HAPS (horizontal offset ~0), so this leg does not sweep with distance --
# it is a constant ~19.5 km slant range (20 km - 0.5 km altitude difference).
d_relay1_km = slant_dist_km(0.0, HAPS_ALT_M, UAV_ALT_M)

# UAV -> UE is the leg that reaches outward as the UAV relays toward the UE.
d_relay2_km = slant_dist_km(r_horiz_km, UAV_ALT_M, UE_ALT_M)     # UAV -> UE

fspl_2ghz = fspl_db(f_2ghz, d_service_km)                        # HAPS -> UE (2 GHz, service link)
fspl_38ghz = np.full_like(r_horiz_km, fspl_db(f_38ghz, d_relay1_km))  # HAPS -> UAV (38 GHz, relay link, 1st hop, fixed)
fspl_uav_ue = fspl_db(f_2ghz, d_relay2_km)                       # UAV -> UE (2 GHz, relay link, 2nd hop)
fspl_relay_total = fspl_38ghz + fspl_uav_ue   # Combined HAPS -> UAV -> UE, dB sum (illustrative only -- see notes below)

# Indicative-only reference: 38 GHz FSPL if it were swept over the full 0-100 km
# range using the *service-link* geometry (HAPS -> ground point). The 38 GHz
# hop never actually extends past the UAV (~19.5 km) -- this line exists only
# to visualize the constant frequency-scaling offset vs. the 2 GHz service link.
fspl_38ghz_ref = fspl_db(f_38ghz, d_service_km)

fspl_diff = fspl_38ghz_ref - fspl_2ghz
fspl_diff_at_100km = fspl_diff[-1]

print(f"Fixed HAPS->UAV (38 GHz) slant distance: {d_relay1_km:.2f} km, path loss: {fspl_38ghz[0]:.2f} dB")
print(f"FSPL offset at 100 km (38 GHz indicative reference - 2 GHz Service, same geometry): {fspl_diff_at_100km:.2f} dB")
print(f"Theoretical frequency-only offset: 20*log10(38/2) = {20 * np.log10(38 / 2):.2f} dB")

## Generate Figure 1b

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(r_horiz_km, fspl_2ghz, linewidth=2.5, label='2 GHz (Service Link, HAPS-UE)', color='#2E86AB', linestyle='-')
ax.plot(r_horiz_km, fspl_38ghz, linewidth=2.5, label=f'38 GHz (Relay Link, HAPS-UAV, 1st hop, fixed {d_relay1_km:.1f} km)', color='#A23B72', linestyle='--')
ax.plot(r_horiz_km, fspl_uav_ue, linewidth=2.5, label='2 GHz (Relay Link, UAV-UE, 2nd hop)', color='#3B9C4A', linestyle='--')
ax.plot(r_horiz_km, fspl_relay_total, linewidth=2.5, label='Combined Relay Path (HAPS-UAV-UE, hop 1 + hop 2, indicative)', color='#E8792E', linestyle=':')
ax.plot(r_horiz_km, fspl_38ghz_ref, linewidth=1.5, label='38 GHz (indicative full-range reference, same geometry as Service Link)', color='#888888', linestyle='--', alpha=0.7)

ax.set_xlabel('Horizontal Ground Distance (UAV to UE) [km]', fontsize=12, fontweight='bold')
ax.set_ylabel('Free-Space Path Loss [dB]', fontsize=12, fontweight='bold')
ax.set_title('HAPS/UAV Free-Space Path Loss: Frequency Comparison (Eq. 1)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=10, loc='upper left', framealpha=0.95)

mid_point = 50
offset_y_2 = fspl_db(f_2ghz, slant_dist_km(mid_point, HAPS_ALT_M, UE_ALT_M))
offset_y_38ref = fspl_db(f_38ghz, slant_dist_km(mid_point, HAPS_ALT_M, UE_ALT_M))
ax.annotate('', xy=(mid_point, offset_y_38ref), xytext=(mid_point, offset_y_2),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(mid_point + 5, (offset_y_2 + offset_y_38ref) / 2,
        f'{offset_y_38ref - offset_y_2:.2f} dB (at {mid_point} km)', fontsize=10, fontweight='bold', va='center')

feeder_distance_km = 0.5
ax.axvline(x=feeder_distance_km, color='red', linestyle='--', linewidth=2, alpha=0.6,
           label=f'Feeder link range (0-{feeder_distance_km} km, separate short backhaul hop, not plotted)')
ax.text(feeder_distance_km, 175, f'  Feeder link range (0-{feeder_distance_km} km)',
        fontsize=9, color='red', weight='bold', va='top')

ax.set_ylim([60, 300])
ax.set_xlim([0, 100])
plt.tight_layout()
plt.savefig('../output/1b_haps_fspl_2ghz_vs_38ghz.png', dpi=300, bbox_inches='tight')
print('[OK] Figure saved: ../output/1b_haps_fspl_2ghz_vs_38ghz.png')
plt.show()
plt.close(fig)

## Notes on interpreting the Relay-Link curves

- **38 GHz (Relay Link, HAPS-UAV, 1st hop)** is drawn **dashed** and as a flat
  line because it is a fixed, near-vertical ~19.5 km hop -- it does not
  physically extend past the UAV, so the flat line across the 0-100 km axis
  is indicative only (it shows the hop's constant loss for reference against
  the other, distance-varying curves), not a link that itself reaches out to
  100 km.
- **38 GHz (indicative full-range reference)** is a purely illustrative
  curve: 38 GHz FSPL computed with the *service-link* geometry (as if the
  HAPS transmitted 38 GHz all the way to the ground). No such link exists in
  the system model -- it is included only to visualize the constant
  ~25.6 dB frequency-scaling offset ($20\log_{10}(38/2)$) against the 2 GHz
  service link, confirming the free-space model behaves consistently across
  bands.
- **Combined Relay Path (HAPS-UAV-UE)** is the dB **sum** of the two relay
  hops' path losses. This is an FSPL-only, illustrative quantity -- it is
  *not* a valid end-to-end link budget, because the two hops are independent
  RF links (separate transmit power, noise floor, and decode/re-encode at
  the UAV), not losses that stack in series within one continuous chain.
  Consequently this curve should not be read as "the relay path is worse
  than the direct service link" -- a proper relay-vs-direct comparison
  requires comparing per-hop SINR/capacity, with the two-hop
  decode-and-forward relay bottlenecked by the *weaker* of the two hops
  (`min`, not sum), as implemented in `generate_relay_table.py`'s
  `relay_two_hop_capacity_bps_hz()`. This figure shows only the underlying
  FSPL (Eq. 1) building blocks, not achievable throughput.

## Summary

In [ ]:
print('=' * 70)
print('Figure 1b Summary')
print('=' * 70)
print('HAPS Altitude: 20 km, UAV Altitude: 500 m AGL, UE Altitude: 1.5 m')
print('Horizontal Ground Distance Range: 0-100 km, measured UAV to UE (3D slant distance used in FSPL)')
print('Curve 1 (Service Link, HAPS->UE): 2 GHz, sweeps 0-100 km')
print(f'Curve 2 (Relay Link, HAPS->UAV, 1st hop): 38 GHz, fixed at {d_relay1_km:.2f} km (near-vertical, UAV below HAPS)')
print('Curve 3 (Relay Link, UAV->UE, 2nd hop): 2 GHz, sweeps 0-100 km')
print('Curve 4 (Combined Relay Path, HAPS->UAV->UE): hop 1 (fixed) + hop 2 (swept), dB sum, indicative only')
print('Curve 5 (38 GHz indicative full-range reference): same geometry as Service Link, for offset visualization only')
print(f'FSPL, Curve 1 (Service, HAPS->UE) at 0 km:        {fspl_2ghz[0]:.2f} dB')
print(f'FSPL, Curve 2 (Relay, HAPS->UAV), fixed:          {fspl_38ghz[0]:.2f} dB')
print(f'FSPL, Curve 3 (Relay, UAV->UE) at 0 km:           {fspl_uav_ue[0]:.2f} dB')
print(f'FSPL, Curve 4 (Combined Relay Path) at 0 km:      {fspl_relay_total[0]:.2f} dB')
print(f'FSPL, Curve 5 (38 GHz indicative ref) at 0 km:    {fspl_38ghz_ref[0]:.2f} dB')
print(f'FSPL, Curve 1 (Service, HAPS->UE) at 100 km:      {fspl_2ghz[-1]:.2f} dB')
print(f'FSPL, Curve 3 (Relay, UAV->UE) at 100 km:         {fspl_uav_ue[-1]:.2f} dB')
print(f'FSPL, Curve 4 (Combined Relay Path) at 100 km:    {fspl_relay_total[-1]:.2f} dB')
print(f'FSPL, Curve 5 (38 GHz indicative ref) at 100 km:  {fspl_38ghz_ref[-1]:.2f} dB')
print('=' * 70)